# Cyber Asset Exclusion - EDA and Leakage-Safe Baseline
Run after placing training/validation/raw files on the EC2/Jupyter instance.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('src').resolve()))
import pandas as pd, yaml
from cyber_exclusion.io import read_csv
from cyber_exclusion.dataset import build_feature_frame
from cyber_exclusion.model import train_grouped_ensemble

In [ ]:
TRAIN='/data/training_data.csv'
RAW_ROOT='/data/raw'
CFG='config/default.yaml'
train_rows=read_csv(TRAIN)
train_rows.head(), train_rows.shape

In [ ]:
with open(CFG) as f: cfg=yaml.safe_load(f)
features=build_feature_frame(train_rows, RAW_ROOT, '.cache/raw', cfg.get('aws_region'))
features[['client_id','scan_id','asset','asset_type','raw_available','scan_outlier_score']].head()

In [ ]:
label=[c for c in cfg['label_candidates'] if c in features.columns][0]
print(features[label].value_counts(dropna=False))
print('clients', features.client_id.nunique(), 'scans', features.scan_id.nunique(), 'raw coverage', features.raw_available.mean())

In [ ]:
bundle,oof,metrics=train_grouped_ensemble(features,cfg)
metrics

In [ ]:
oof.sort_values('p_exclude', ascending=False).head(30)